# 14 · MCP：外部资源为何仍要证明来源

外部工具调用成功，只说明某一次绑定收到了结果。随后读取 UI 资源时，还要证明 URI 属于那次调用、账户仍匹配、连接/权限变化没有让旧结果越权。
本章用内存中的异步 provider 返回资源字节，真实运行来源账本和取消逻辑；它不是 MCP 握手、OAuth 或浏览器渲染器。
关键取舍是把资源来源绑定到调用前捕获的 descriptor，同时在接受返回值前重查当前授权代次。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [S16 · mcp_tool_metadata / get_mcp_app_resource_uri](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/mcp_tool_call.rs#L2155-L2222)：descriptor URI 三个键有优先次序；仅 inline/fullscreen 生成显式 UI，URI 可单独保留。
- [S21 · ResourceOrigin::read](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/codex-mcp/src/resource_origin.rs#L249-L317)：读取前核对 URI、歧义账户、当前 connector/link 与政策，再通过绑定读取。
- [S10 · RequestScope / RequestCancellation / CancelOnDrop](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/rmcp-client/src/request_cancellation.rs#L21-L131)：取消作用域跟踪取消写出；OnDrop 尝试发送通知。


## 实验 1 · descriptor 投影与成功来源账本


UI URI 来自工具目录 descriptor，不能从任意工具正文升级为可信路由。教学版进一步把 URI 限为 ui://，比固定源码提取规则更严格。
先捕获绑定，再标记调用成功，资源读取才取得来源。


In [2]:
bridge = load_example('14', 'ui_bridge')
import asyncio
from dataclasses import replace
URI = 'ui://task-list/report'
binding = bridge.Binding('task-service', 'inspect', 'account-A', 1, 1)
descriptor = {'_meta': {'ui': {'resourceUri': URI},
    'openai/ui': {'preferredModelDisplayMode': 'inline'}}}
ledger = bridge.OriginLedger()
origin = ledger.prepare('call-1', binding, descriptor)
with expect.assertRaisesRegex(bridge.BridgeError, 'no_successful_origin'):
    ledger.lookup('call-1')
ledger.complete('call-1', succeeded=True)
assert ledger.lookup('call-1').projection.uri == URI
assert bridge.project_descriptor({'content': [{'text': URI}]}).uri is None
with expect.assertRaisesRegex(bridge.BridgeError, 'invalid_ui_uri'):
    bridge.project_descriptor({'_meta': {'ui': {'resourceUri': 'https://untrusted.invalid/ui'}}})
print(origin)


Origin(call_id='call-1', binding=Binding(server='task-service', tool='inspect', account='account-A', connection_epoch=1, catalog_version=1), projection=UiProjection(uri='ui://task-list/report', display_mode='inline'))


## 实验 2 · 读取必须同时匹配调用、服务器与 URI


同一个 URI 字符串可能被不同服务使用，所以 URI 不能单独做授权凭据。provider 返回值也要再次核对来源与媒体类型。
观察被拒的请求是否真的进入 provider；调用前拒绝应保持计数不变。


In [3]:
requests = []
async def fetch(current_binding, uri):
    requests.append((current_binding, uri))
    return bridge.Resource(current_binding.server, uri, bridge.UI_MIME, b'<p>task report</p>')
authority = bridge.Authority(binding)
reader = bridge.ScopedResourceReader(ledger, authority, fetch)
resource = await reader.read('read-1', 'call-1', binding.server, URI)
assert resource.body == b'<p>task report</p>'
with expect.assertRaisesRegex(bridge.BridgeError, 'origin_mismatch'):
    await reader.read('wrong-server', 'call-1', 'another-service', URI)
with expect.assertRaisesRegex(bridge.BridgeError, 'origin_mismatch'):
    await reader.read('wrong-uri', 'call-1', binding.server, URI+'/other')
assert len(requests) == 1
assert reader.pending == {}
print({'provider_calls': len(requests), 'bytes_received': len(resource.body)})


{'provider_calls': 1, 'bytes_received': 18}


## 实验 3 · await 期间权限变化使旧返回值失效


请求已出发时，账户可以被切换或连接被替换。发送前检查不足以保障接收时仍有效。
用 Event 精确控制竞态，不靠 sleep 猜时序：先进入 provider，再改 authority，最后释放返回值。


In [4]:
entered, release = asyncio.Event(), asyncio.Event()
async def delayed_fetch(current_binding, uri):
    entered.set()
    await release.wait()
    return await fetch(current_binding, uri)
racing = bridge.ScopedResourceReader(ledger, authority, delayed_fetch)
task = asyncio.create_task(racing.read('race', 'call-1', binding.server, URI))
await asyncio.wait_for(entered.wait(), 2)
authority.replace(replace(binding, connection_epoch=2))
release.set()
with expect.assertRaisesRegex(bridge.BridgeError, 'stale_authority'):
    await asyncio.wait_for(task, 2)
assert racing.pending == {}
# 相同账户重新读取使用当前连接；不是接受旧连接的迟到结果。
recovered = await reader.read('fresh-after-reconnect', 'call-1', binding.server, URI)
assert requests[-1][0].connection_epoch == 2
print({'recovered_bytes': len(recovered.body), 'current_epoch': requests[-1][0].connection_epoch})


{'recovered_bytes': 18, 'current_epoch': 2}


## 实验 4 · provider 忽略取消，宿主仍拒绝迟到结果


取消能结束本地等待，不保证远端动作停止。这个 provider 故意吞掉取消并返回数据；reader 必须依据本地 waiter 的取消标志拒绝它。
与之相对，已经完成的结果不能因之后一次无效 cancel 被改写。


In [5]:
entered = asyncio.Event()
async def stubborn_fetch(current_binding, uri):
    entered.set()
    try:
        await asyncio.Event().wait()
    except asyncio.CancelledError:
        return await fetch(current_binding, uri)
cancellable = bridge.ScopedResourceReader(ledger, authority, stubborn_fetch)
task = asyncio.create_task(cancellable.read('cancel-me', 'call-1', binding.server, URI))
await asyncio.wait_for(entered.wait(), 2)
assert cancellable.cancel('cancel-me')
with expect.assertRaises(asyncio.CancelledError):
    await asyncio.wait_for(task, 2)
assert cancellable.pending == {}
assert not cancellable.cancel('cancel-me')
if cancellable.retiring:
    await asyncio.gather(*tuple(cancellable.retiring), return_exceptions=True)
print('provider 已执行返回分支，但取消后的本地等待未接受资源。')


provider 已执行返回分支，但取消后的本地等待未接受资源。


## 实验 5 · 重连后的数字 ID 相同，身份仍不同


请求键加入 server、连接 epoch、方向与数字序号。断连清空的是本连接等待表，返回 unknown_calls；
它既不能证明外部副作用未发生，也不能授权自动重试同一写操作。


In [6]:
pending = bridge.PendingBook('task-service')
old_key = pending.begin('write-call-1')
unknown = pending.disconnect()
new_key = pending.begin('read-call-2')
assert old_key[-1] == new_key[-1] == 1
assert old_key != new_key and unknown == ('write-call-1',)
assert pending.resolve(old_key) is None
assert pending.resolve(new_key) == 'read-call-2'
assert pending.resolve(new_key) is None
print({'old_key': old_key, 'new_key': new_key, 'unconfirmed_calls': unknown})


{'old_key': ('task-service', 1, 'client_to_server', 1), 'new_key': ('task-service', 2, 'client_to_server', 1), 'unconfirmed_calls': ('write-call-1',)}


## 你现在可以解释什么

能解释为何 descriptor、调用来源、当前授权和连接代次缺一不可。这里没有真实 MCP server、OAuth、网络重试、浏览器 UI 或真实远端取消效果。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
